# StretchTime ETTh1 forecast visualization

This notebook trains or loads the **StretchTime** model and saves simple forecast visualizations for the selected dataset:

- one visual per channel
- each visual shows input history, ground truth future, and predicted future
- metrics shown in titles are computed in the same standardized scale used by the training/test loop

By default, the notebook is configured for `ETT-small/ETTh1.csv` with `SEQ_LEN = 96` and `PRED_LEN = 96`.

If `LOAD_CHECKPOINT_FOR_VIS_ONLY = "pt_model_96_96.pth"`, the notebook skips training and only generates visualizations from that checkpoint. Outputs are written to `visualizations/<setting_name>/`.


## Setup and imports

These cells configure local paths, import the copied StretchTime/data utilities, and apply small compatibility patches needed for the notebook environment.


In [ ]:
# Environment setup
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "dataset"

# Matplotlib tries to write to ~/.matplotlib on some machines; keep caches local/writable.
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib-cache"))
os.makedirs(os.environ["MPLCONFIGDIR"], exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATASET_ROOT}")


In [ ]:
# Imports and compatibility patches
import copy
import random
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# The bundled data loaders use the old pandas DataFrame.drop(labels, axis) positional form.
# Pandas 2.x removed that positional axis argument, so patch it in-notebook without editing the modules.
_ORIGINAL_DF_DROP = pd.DataFrame.drop
if not getattr(pd.DataFrame.drop, "_stretchtime_compat", False):
    def _drop_compat(self, labels=None, axis=0, index=None, columns=None, level=None, inplace=False, errors="raise"):
        return _ORIGINAL_DF_DROP(
            self,
            labels=labels,
            axis=axis,
            index=index,
            columns=columns,
            level=level,
            inplace=inplace,
            errors=errors,
        )
    _drop_compat._stretchtime_compat = True
    pd.DataFrame.drop = _drop_compat

from data_provider.data_factory import data_provider
from exp.exp_main import Exp_Main
from models.StretchTime import Model as StretchTimeModel, SymplecticPE, apply_J
from utils.metrics import metric

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


## Dataset discovery and script hyperparameters

This section maps dataset names to the local `dataset/` files and stores the StretchTime script hyperparameters used for matching the original training setup.


In [ ]:
# Dataset discovery and configuration helpers
SCRIPT_DEFAULTS = {
    "ETT-small/ETTh1.csv": {"alias": "ETTh1", "data": "ETTh1", "freq": "h"},
    "ETT-small/ETTh2.csv": {"alias": "ETTh2", "data": "ETTh2", "freq": "h"},
    "ETT-small/ETTm1.csv": {"alias": "ETTm1", "data": "ETTm1", "freq": "t"},
    "ETT-small/ETTm2.csv": {"alias": "ETTm2", "data": "ETTm2", "freq": "t"},
    "weather/weather.csv": {"alias": "Weather", "data": "custom", "freq": "h"},
    "electricity/electricity.csv": {"alias": "Electricity", "data": "custom", "freq": "h"},
    "traffic/traffic.csv": {"alias": "Traffic", "data": "custom", "freq": "h"},
    "exchange_rate.csv": {"alias": "Exchange", "data": "custom", "freq": "d"},
    "national_illness.csv": {"alias": "ILI", "data": "custom", "freq": "w"},
    "national_illness_full.csv": {"alias": "ILI_full", "data": "custom", "freq": "w"},
    "Solar/solar_AL.txt": {"alias": "Solar", "data": "Solar", "freq": "h"},
    "PEMS/PEMS03.npz": {"alias": "PEMS03", "data": "PEMS", "freq": "h"},
    "PEMS/PEMS04.npz": {"alias": "PEMS04", "data": "PEMS", "freq": "h"},
    "PEMS/PEMS07.npz": {"alias": "PEMS07", "data": "PEMS", "freq": "h"},
    "PEMS/PEMS08.npz": {"alias": "PEMS08", "data": "PEMS", "freq": "h"},
}

SUPPORTED_SUFFIXES = {".csv", ".txt", ".npz"}
M4_PATTERNS = ["Yearly", "Quarterly", "Monthly", "Weekly", "Daily", "Hourly"]


SCRIPT_HPARAMS = {
    "ETTh1": {
        96: dict(e_layers=2, endog_dim_factor=8, exog_dim_factor=8, weight_decay=0.1, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        192: dict(e_layers=2, endog_dim_factor=8, exog_dim_factor=8, weight_decay=0.1, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        336: dict(e_layers=2, endog_dim_factor=8, exog_dim_factor=8, weight_decay=0.1, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        720: dict(e_layers=2, endog_dim_factor=8, exog_dim_factor=8, weight_decay=0.1, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=16, gradient_accumulation=2),
    },
    "ETTh2": {
        96: dict(e_layers=2, endog_dim_factor=32, exog_dim_factor=32, weight_decay=0.1, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        192: dict(e_layers=2, endog_dim_factor=32, exog_dim_factor=32, weight_decay=0.1, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        336: dict(e_layers=2, endog_dim_factor=8, exog_dim_factor=8, weight_decay=0.1, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        720: dict(e_layers=2, endog_dim_factor=8, exog_dim_factor=8, weight_decay=0.1, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=16, gradient_accumulation=2),
    },
    "ETTm1": {
        96: dict(e_layers=3, endog_dim_factor=64, exog_dim_factor=64, weight_decay=0, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        192: dict(e_layers=2, endog_dim_factor=64, exog_dim_factor=64, weight_decay=0, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        336: dict(e_layers=2, endog_dim_factor=64, exog_dim_factor=64, weight_decay=0, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        720: dict(e_layers=2, endog_dim_factor=96, exog_dim_factor=96, weight_decay=0, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=4, gradient_accumulation=8),
    },
    "ETTm2": {
        96: dict(e_layers=2, endog_dim_factor=64, exog_dim_factor=64, weight_decay=0, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        192: dict(e_layers=2, endog_dim_factor=64, exog_dim_factor=48, weight_decay=0, random_channel_dropout=1, learning_rate=0.00035, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        336: dict(e_layers=3, endog_dim_factor=64, exog_dim_factor=64, weight_decay=0, random_channel_dropout=1, learning_rate=0.0005, n_heads=4, pct_start=0, batch_size=32, gradient_accumulation=1),
        720: dict(e_layers=2, endog_dim_factor=96, exog_dim_factor=96, weight_decay=0, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=4, gradient_accumulation=8),
    },
    "Weather": {
        96: dict(e_layers=3, endog_dim_factor=64, exog_dim_factor=64, weight_decay=0, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        192: dict(e_layers=3, endog_dim_factor=48, exog_dim_factor=48, weight_decay=0, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        336: dict(e_layers=3, endog_dim_factor=64, exog_dim_factor=64, weight_decay=0, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=32, gradient_accumulation=1),
        720: dict(e_layers=3, endog_dim_factor=96, exog_dim_factor=96, weight_decay=0, random_channel_dropout=1, learning_rate=0.00035, n_heads=8, pct_start=0, batch_size=8, gradient_accumulation=4),
    },
    "Electricity": {pred_len: dict(e_layers=3, endog_dim_factor=128, exog_dim_factor=128, weight_decay=0, random_channel_dropout=0, learning_rate=0.0005, n_heads=4, pct_start=0, batch_size=8, gradient_accumulation=4) for pred_len in [96, 192, 336, 720]},
    "Solar": {
        96: dict(e_layers=3, endog_dim_factor=128, exog_dim_factor=128, weight_decay=0, random_channel_dropout=0, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=8, gradient_accumulation=4),
        192: dict(e_layers=3, endog_dim_factor=128, exog_dim_factor=128, weight_decay=0, random_channel_dropout=0, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=8, gradient_accumulation=4),
        336: dict(e_layers=3, endog_dim_factor=128, exog_dim_factor=128, weight_decay=0, random_channel_dropout=0, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=2, gradient_accumulation=16),
        720: dict(e_layers=3, endog_dim_factor=128, exog_dim_factor=128, weight_decay=0, random_channel_dropout=0, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=2, gradient_accumulation=16),
    },
    "PEMS03": {pred_len: dict(e_layers=3, endog_dim_factor=128, exog_dim_factor=128, weight_decay=0, random_channel_dropout=0, learning_rate=0.0005, n_heads=4, pct_start=0, batch_size=8, gradient_accumulation=4) for pred_len in [12, 24, 48, 96]},
    "PEMS04": {pred_len: dict(e_layers=3, endog_dim_factor=128, exog_dim_factor=128, weight_decay=0, random_channel_dropout=0, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=2, gradient_accumulation=16) for pred_len in [12, 24, 48, 96]},
    "PEMS08": {
        12: dict(e_layers=2, endog_dim_factor=128, exog_dim_factor=128, weight_decay=0, random_channel_dropout=0, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=2, gradient_accumulation=16),
        24: dict(e_layers=3, endog_dim_factor=128, exog_dim_factor=128, weight_decay=0, random_channel_dropout=0, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=2, gradient_accumulation=16),
        48: dict(e_layers=3, endog_dim_factor=128, exog_dim_factor=128, weight_decay=0, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=4, gradient_accumulation=8),
        96: dict(e_layers=3, endog_dim_factor=128, exog_dim_factor=128, weight_decay=0, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=4, gradient_accumulation=8),
    },
}


def script_hparams(alias, pred_len):
    if alias in SCRIPT_HPARAMS and pred_len in SCRIPT_HPARAMS[alias]:
        return SCRIPT_HPARAMS[alias][pred_len]
    return dict(e_layers=2, endog_dim_factor=8, exog_dim_factor=8, weight_decay=0.1, random_channel_dropout=1, learning_rate=0.0005, n_heads=8, pct_start=0, batch_size=BATCH_SIZE, gradient_accumulation=1)


def list_available_datasets(dataset_root=DATASET_ROOT):
    paths = []
    for path in sorted(Path(dataset_root).rglob("*")):
        if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES and not path.name.startswith("."):
            # Raw tsf files are not wired to this repo's data_provider; M4 is exposed below as m4:<pattern>.
            if "m4" in path.relative_to(dataset_root).parts or "tsf" in path.relative_to(dataset_root).parts:
                continue
            paths.append(path.relative_to(dataset_root).as_posix())
    if (Path(dataset_root) / "m4" / "training.npz").exists():
        paths.extend([f"m4:{pattern}" for pattern in M4_PATTERNS])
    return paths


def _safe_csv_columns(path):
    try:
        return list(pd.read_csv(path, nrows=5).columns)
    except Exception:
        return []


def infer_dataset_config(data_path, dataset_root=DATASET_ROOT):
    # Infer StretchTime data-loader args for a dataset path under dataset/.
    rel = Path(data_path).as_posix()
    if rel.startswith("m4:"):
        pattern = rel.split(":", 1)[1]
        return {
            "data_path": rel,
            "data": "m4",
            "alias": f"M4_{pattern}",
            "enc_in": 1,
            "target": "OT",
            "freq": "h",
            "root_path": str(Path(dataset_root) / "m4"),
            "seasonal_patterns": pattern,
        }

    full_path = Path(dataset_root) / rel
    defaults = copy.deepcopy(SCRIPT_DEFAULTS.get(rel, {}))
    suffix = full_path.suffix.lower()

    if suffix == ".npz":
        with np.load(full_path, allow_pickle=True) as data:
            enc_in = int(data["data"].shape[1])
        data_type = defaults.get("data", "PEMS")
        target = "OT"
    elif suffix == ".txt":
        first = full_path.read_text().splitlines()[0]
        enc_in = len(first.split(","))
        data_type = defaults.get("data", "Solar")
        target = "OT"
    else:
        cols = _safe_csv_columns(full_path)
        lower_cols = {c.lower(): c for c in cols}
        date_col = lower_cols.get("date", cols[0] if cols else "date")
        numeric_cols = [c for c in cols if c != date_col]
        enc_in = max(1, len(numeric_cols))
        target = defaults.get("target", numeric_cols[-1] if numeric_cols else "OT")
        if rel in SCRIPT_DEFAULTS:
            data_type = defaults.get("data", "custom")
        elif Path(rel).name.startswith("ETTh"):
            data_type = Path(rel).stem
        elif Path(rel).name.startswith("ETTm"):
            data_type = Path(rel).stem
        else:
            data_type = "custom"

    alias = defaults.get("alias", Path(rel).stem.replace(".", "_"))
    freq = defaults.get("freq", "h")
    return {"data_path": rel, "data": data_type, "alias": alias, "enc_in": enc_in, "target": target, "freq": freq, "root_path": str(dataset_root), "seasonal_patterns": "Monthly"}


def print_datasets():
    rows = []
    for rel in list_available_datasets():
        cfg = infer_dataset_config(rel)
        rows.append({"dataset": rel, "loader": cfg["data"], "channels": cfg["enc_in"], "target": cfg["target"], "freq": cfg["freq"]})
    return pd.DataFrame(rows)

print_datasets()


## Run configuration

Set the dataset, sequence length, prediction length, epochs, and checkpoint-only mode here. For the current run, keep `DATASET = "ETT-small/ETTh1.csv"`.


In [ ]:
# Choose dataset/training parameters
# Pick any path shown by print_datasets(), relative to ./dataset.
DATASET = "ETT-small/ETTh1.csv"

SEQ_LEN = 96
PRED_LEN = 96          # PEMS commonly uses 12/24/48/96; most other scripts use 96/192/336/720.
TRAIN_EPOCHS = 100     # Matches scripts/StretchTime/*.sh defaults.
BATCH_SIZE = 32
LEARNING_RATE = 5e-4
PATIENCE = 12
RANDOM_SEED = 2026

# Keep this off in notebooks unless you specifically want torch.compile startup cost.
TORCH_COMPILE = 1
USE_AMP = torch.cuda.is_available()

# Set to "pt_model_96_96.pth" to skip training and only generate notebook PNG visualizations.
LOAD_CHECKPOINT_FOR_VIS_ONLY = None


In [ ]:
# Learned-clock probe for StretchTime's SymplecticPE
_ORIGINAL_RPE = SymplecticPE.rpe


def install_clock_probe():
    # Monkey-patch SymplecticPE.rpe so each module stores its latest learned clock.
    if getattr(SymplecticPE.rpe, "_stretchtime_clock_probe", False):
        return

    def _rpe_with_clock_probe(self, q, k, clock_x=None):
        if self.learnable_clock:
            assert clock_x is not None, "When learnable_clock=True, pass clock_x of shape (B, L, H*D)."
            clock_delta = F.softplus(self.clock(clock_x).reshape(q.size(0), q.size(1), self.H, -1))
            clock = clock_delta.cumsum(dim=1)
        else:
            clock_delta = None
            clock = None

        self.last_clock = None if clock is None else clock.detach().cpu()
        self.last_clock_delta = None if clock_delta is None else clock_delta.detach().cpu()

        q_pos = self.forward(q, is_key=False, transpose=False, clock=clock)
        k_pos = self.forward(k, is_key=True, transpose=False, clock=clock)
        return q_pos, k_pos

    _rpe_with_clock_probe._stretchtime_clock_probe = True
    SymplecticPE.rpe = _rpe_with_clock_probe


def collect_clock_summary(model):
    # Return averaged learned clock and speed from all StretchTime attention layers.
    clocks = []
    speeds = []
    names = []
    bare_model = model.module if hasattr(model, "module") else model
    for name, module in bare_model.named_modules():
        if isinstance(module, SymplecticPE) and getattr(module, "last_clock", None) is not None:
            # [batch_or_channel, time, heads, blocks] -> [time]
            clocks.append(module.last_clock.float().mean(dim=(0, 2, 3)).numpy())
            speeds.append(module.last_clock_delta.float().mean(dim=(0, 2, 3)).numpy())
            names.append(name)
    if not clocks:
        raise RuntimeError("No learned clocks captured. Run an evaluation forward pass after install_clock_probe().")
    return {
        "layer_names": names,
        "clock_by_layer": np.stack(clocks),
        "speed_by_layer": np.stack(speeds),
        "clock_mean": np.stack(clocks).mean(axis=0),
        "speed_mean": np.stack(speeds).mean(axis=0),
    }

install_clock_probe()
print("Clock probe installed.")


## Training or checkpoint loading

This cell builds StretchTime arguments, applies the dataset/horizon hyperparameters, and either trains the model with `Exp_Main.train()` or loads `LOAD_CHECKPOINT_FOR_VIS_ONLY` for visualization-only runs.


In [ ]:
# Build args and train StretchTime

def set_seed(seed=2026):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_args(data_path=DATASET, seq_len=SEQ_LEN, pred_len=PRED_LEN, train_epochs=TRAIN_EPOCHS, batch_size=BATCH_SIZE):
    cfg = infer_dataset_config(data_path)
    enc_in = int(cfg["enc_in"])
    number_of_targets = enc_in
    time_emb_dim = 0 if cfg["data"] == "m4" else 4
    hp = script_hparams(cfg["alias"], pred_len)
    model_id = f"notebook_StretchTime_{cfg['alias']}_sl{seq_len}_pl{pred_len}"
    return SimpleNamespace(
        random_seed=RANDOM_SEED,
        is_training=1,
        model_id=model_id,
        model="StretchTime",
        model_type="DL",
        data=cfg["data"],
        root_path=str(cfg.get("root_path", DATASET_ROOT)) + "/",
        data_path=cfg["data_path"],
        features="M",
        target=cfg["target"],
        freq=cfg["freq"],
        checkpoints=str(PROJECT_ROOT / "checkpoints"),
        scale=1,
        train_ratio=0.7,
        test_ratio=0.2,
        plot_every=0,
        iterative_prediction=0,
        number_of_targets=number_of_targets,
        seq_len=seq_len,
        label_len=0,
        pred_len=pred_len,
        fc_dropout=0.05,
        head_dropout=0.0,
        patch_len=16,
        stride=8,
        padding_patch="end",
        revin=1,
        affine=0,
        subtract_last=0,
        decomposition=0,
        kernel_size=25,
        individual=0,
        F_conv_output=8,
        F_noconv_output=8,
        F_gconv_output_rate=1,
        F_lin_output=16,
        F_id_output_rate=1,
        F_emb_output=4,
        continuity_beta=1.0,
        channel_sparsity=1,
        temporal_sparsity=1,
        output_projection=1,
        predictor_dropout=0.1,
        mid_gate=1,
        split_ratio=1,
        embed_type=0,
        enc_in=enc_in,
        dec_in=enc_in,
        c_out=enc_in,
        d_model=0,
        n_heads=hp["n_heads"],
        e_layers=hp["e_layers"],
        d_layers=1,
        d_ff=2048,
        moving_avg=25,
        factor=1,
        distil=True,
        dropout=0.0,
        embed="timeF",
        activation="gelu",
        output_attention=False,
        do_predict=False,
        time_emb_dim=time_emb_dim,
        mlp_ratio=8,
        num_workers=6,
        itr=1,
        train_epochs=train_epochs,
        batch_size=hp["batch_size"],
        patience=PATIENCE,
        learning_rate=hp["learning_rate"],
        des="Notebook",
        loss="mse",
        lradj="TST",
        pct_start=float(hp["pct_start"]),
        use_amp=USE_AMP,
        max_grad_norm=1,
        test_every=0,
        gradient_accumulation=hp["gradient_accumulation"],
        batch_size_test=0,
        log_every=20,
        predictor="StretchTime",
        resume="none",
        compile=TORCH_COMPILE,
        use_wandb=0,
        wandb_project="StretchTime-CAPS",
        use_gpu=torch.cuda.is_available(),
        gpu=0,
        use_multi_gpu=False,
        devices="0",
        device_ids=[0],
        test_flop=False,
        random_drop_training=0,
        model_label="StretchTime",
        univariate=0,
        channel_independence=1,
        decomp_method="dft_decomp",
        use_norm=1,
        down_sampling_layers=2,
        down_sampling_window=2,
        down_sampling_method="avg",
        channel_mixing=1,
        percent=100,
        seasonal_patterns=cfg.get("seasonal_patterns", "Monthly"),
        inverse=False,
        task_name="long_term_forecast",
        top_k=3,
        num_kernels=6,
        use_future_temporal_feature=0,
        random_channel_dropout=hp["random_channel_dropout"],
        weight_decay=hp["weight_decay"],
        endog_dim_factor=hp["endog_dim_factor"],
        exog_dim_factor=hp["exog_dim_factor"],
    )


def setting_name(args, run_idx=0):
    return '{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}'.format(
        args.model_id, args.model, args.data, args.features, args.seq_len, args.label_len, args.pred_len,
        args.d_model, args.n_heads, args.e_layers, args.d_layers, args.d_ff, args.factor,
        args.embed, args.distil, args.des, run_idx
    )


def unpack_model_output(outputs):
    return outputs[0] if isinstance(outputs, tuple) else outputs


def select_target_window(args, outputs, batch_y):
    f_dim = -args.number_of_targets
    if args.features == "MS":
        f_dim = -1
    outputs = outputs[:, -args.pred_len:, f_dim:]
    targets = batch_y[:, -args.pred_len:, f_dim:]
    return outputs, targets


def make_decoder_input(args, batch_y, device):
    dec_inp = torch.zeros_like(batch_y[:, -args.pred_len:, :]).float()
    dec_inp = torch.cat([batch_y[:, :args.label_len, :], dec_inp], dim=1)
    return dec_inp.float().to(device)


def forward_batch(model, args, batch, device):
    batch_x, batch_y, batch_x_mark, batch_y_mark = [item.float().to(device, non_blocking=True) for item in batch[:4]]
    dec_inp = make_decoder_input(args, batch_y, device)
    outputs = model(batch_x, batch_x_mark, dec_inp, batch_y_mark)
    outputs = unpack_model_output(outputs)
    outputs, targets = select_target_window(args, outputs, batch_y)
    return outputs, targets, batch_x, batch_y


def evaluate_model(model, args, loader, device, max_batches=None):
    model.eval()
    criterion = nn.MSELoss()
    losses, preds, trues = [], [], []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break
            outputs, targets, _, _ = forward_batch(model, args, batch, device)
            loss = criterion(outputs, targets)
            losses.append(float(loss.item()))
            preds.append(outputs.detach().cpu().float().numpy())
            trues.append(targets.detach().cpu().float().numpy())
    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    mae, mse, rmse, mape, mspe, rse, corr = metric(preds, trues)
    return float(np.mean(losses)), {"MSE": mse, "MAE": mae, "RMSE": rmse, "MAPE": mape, "MSPE": mspe}


def train_stretchtime(data_path=DATASET):
    set_seed(RANDOM_SEED)
    install_clock_probe()
    args = build_args(data_path)
    exp = Exp_Main(args)
    setting = setting_name(args)

    if LOAD_CHECKPOINT_FOR_VIS_ONLY:
        checkpoint_path = Path(LOAD_CHECKPOINT_FOR_VIS_ONLY)
        print(f"Skipping training; loading checkpoint for visualization: {checkpoint_path}")
        state_dict = torch.load(checkpoint_path, map_location=exp.device)
        cleaned = {key[7:] if key.startswith("module.") else key: value for key, value in state_dict.items()}
        target_model = getattr(exp.model, "_orig_mod", exp.model)
        target_model.load_state_dict(cleaned, strict=False)
        return exp, args, setting, exp.model

    print(f"Training setting: {setting}")
    model = exp.train(setting)
    return exp, args, setting, model

# Run this cell to train. Re-run with a different DATASET/PRED_LEN as needed.
exp, args, setting, model = train_stretchtime(DATASET)


## Forecast visualizations

This cell runs one test batch through the trained/loaded model and saves a separate PNG for every output channel. Each figure contains only history, ground truth, and prediction.


In [ ]:
# Save simple forecast visualization: history + ground truth vs prediction

VIS_DIR = PROJECT_ROOT / "visualizations" / setting
VIS_DIR.mkdir(parents=True, exist_ok=True)


def _as_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().float().numpy()
    return np.asarray(x)


def metrics_dict(pred, true):
    mae, mse, rmse, mape, mspe, rse, corr = metric(pred, true)
    return {"MSE": mse, "MAE": mae, "RMSE": rmse, "MAPE": mape, "MSPE": mspe}


def inverse_window(dataset, values_2d):
    values_2d = np.asarray(values_2d)
    scaler = getattr(dataset, "scaler", None)
    if scaler is None or not hasattr(scaler, "inverse_transform"):
        return values_2d
    try:
        if values_2d.shape[-1] == getattr(scaler, "n_features_in_", values_2d.shape[-1]):
            return scaler.inverse_transform(values_2d)
    except Exception:
        pass
    return values_2d


def forecast_one_batch(exp, args, split="test", batch_index=0):
    data_set, loader = exp._get_data(flag=split)
    model = exp.model
    device = exp.device
    model.eval()

    chosen = None
    for i, batch in enumerate(loader):
        if i == batch_index:
            chosen = batch
            break
    if chosen is None:
        raise IndexError(f"batch_index={batch_index} is outside {split} loader")

    with torch.no_grad():
        outputs, true_future, batch_x, batch_y = forward_batch(model, args, chosen, device)

    f_dim = -args.number_of_targets
    if args.features == "MS":
        f_dim = -1

    return {
        "dataset": data_set,
        "history": _as_numpy(batch_x[:, :, f_dim:]),
        "true": _as_numpy(true_future),
        "pred": _as_numpy(outputs),
        "standardized_metrics": metrics_dict(_as_numpy(outputs), _as_numpy(true_future)),
    }


def plot_and_save_forecast(result, sample=0, channel=0, output_path=None):
    dataset = result["dataset"]
    hist = inverse_window(dataset, result["history"][sample])
    true = inverse_window(dataset, result["true"][sample])
    pred = inverse_window(dataset, result["pred"][sample])

    selected_standardized_metrics = metrics_dict(
        result["pred"][[sample], :, [channel]],
        result["true"][[sample], :, [channel]],
    )

    seq_len = hist.shape[0]
    pred_len = pred.shape[0]
    t_hist = np.arange(seq_len)
    t_future = np.arange(seq_len, seq_len + pred_len)

    if output_path is None:
        output_path = VIS_DIR / f"{Path(args.data_path).stem}_sample{sample}_channel{channel}_forecast.png"
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(13, 4.5), constrained_layout=True)
    ax.plot(t_hist, hist[:, channel], label="history", color="0.35", linewidth=2)
    ax.plot(t_future, true[:, channel], label="ground truth", color="tab:green", linewidth=2)
    ax.plot(t_future, pred[:, channel], label="prediction", color="tab:orange", linestyle="--", linewidth=2)
    ax.axvline(seq_len - 1, color="black", alpha=0.25, linewidth=1)
    ax.set_title(
        f"{args.data_path} channel {channel} | standardized MSE={selected_standardized_metrics['MSE']:.4f}, "
        f"MAE={selected_standardized_metrics['MAE']:.4f}"
    )
    ax.set_xlabel("time step")
    ax.set_ylabel(f"channel {channel}")
    ax.legend(loc="best")
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved channel {channel} forecast visualization: {output_path}")
    return output_path


def plot_and_save_all_channel_forecasts(result, sample=0):
    n_channels = result["pred"].shape[-1]
    print(f"Saving one forecast plot per channel ({n_channels} channels).")
    output_paths = []
    for channel in range(n_channels):
        output_paths.append(plot_and_save_forecast(result, sample=sample, channel=channel))
    print(f"Batch metrics across all channels, standardized scale: {result['standardized_metrics']}")
    return output_paths


result = forecast_one_batch(exp, args, split="test", batch_index=0)
forecast_paths = plot_and_save_all_channel_forecasts(result, sample=0)

# Display the last channel inline as a quick notebook preview; all channels are saved as PNGs above.
preview_channel = result["pred"].shape[-1] - 1
fig_preview, ax_preview = plt.subplots(figsize=(13, 4.5), constrained_layout=True)
dataset_preview = result["dataset"]
hist_preview = inverse_window(dataset_preview, result["history"][0])
true_preview = inverse_window(dataset_preview, result["true"][0])
pred_preview = inverse_window(dataset_preview, result["pred"][0])
seq_len_preview = hist_preview.shape[0]
t_hist_preview = np.arange(seq_len_preview)
t_future_preview = np.arange(seq_len_preview, seq_len_preview + pred_preview.shape[0])
ax_preview.plot(t_hist_preview, hist_preview[:, preview_channel], label="history", color="0.35", linewidth=2)
ax_preview.plot(t_future_preview, true_preview[:, preview_channel], label="ground truth", color="tab:green", linewidth=2)
ax_preview.plot(t_future_preview, pred_preview[:, preview_channel], label="prediction", color="tab:orange", linestyle="--", linewidth=2)
ax_preview.axvline(seq_len_preview - 1, color="black", alpha=0.25, linewidth=1)
ax_preview.set_title(f"Inline preview: {args.data_path} channel {preview_channel}")
ax_preview.legend(loc="best")
plt.show()


In [ ]:
# Optional: train/visualize several datasets in one loop
# Keep epochs low for smoke tests. For paper-quality runs, increase TRAIN_EPOCHS above.

def run_dataset_suite(datasets, epochs=2, pred_len=96):
    global TRAIN_EPOCHS, PRED_LEN, exp, model, args, setting
    old_epochs, old_pred_len = TRAIN_EPOCHS, PRED_LEN
    TRAIN_EPOCHS, PRED_LEN = epochs, pred_len
    outputs = {}
    try:
        for data_path in datasets:
            print("=" * 90)
            print(f"Dataset: {data_path}")
            exp, args, setting, model = train_stretchtime(data_path)
            result = forecast_one_batch(exp, args, split="test", batch_index=0)
            outputs[data_path] = result
            fig = plot_forecast_and_warp(result, sample=0, channel=-1, title=data_path)
            plt.show()
    finally:
        TRAIN_EPOCHS, PRED_LEN = old_epochs, old_pred_len
    return outputs

# Example:
# suite_results = run_dataset_suite(["ETT-small/ETTh1.csv", "weather/weather.csv", "PEMS/PEMS08.npz"], epochs=2, pred_len=96)
